# Topic 5 - complexity

A difficulty signal per concept (and per lemma where needed), validated
against a graded list. See [`05.54_data_enrich.md`](../../scratch_space/09_concept_model/05.54_data_enrich/05.54_data_enrich.md) Topic 5.

Open questions: which signals predict difficulty; is the concept-level part
enough; test the cross-language hypothesis against Kelly CEFR.

## Setup

Thin caller over the staged cache and the OMW wordnets.

In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import wn
from loguru import logger as lg

from lang_tools.lexicon.ingestion.sources.omw import OMW_LEXICONS, OMW_VERSION
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
paths = get_lang_tools_params().paths
data_fol = paths.data_fol
staging = data_fol / "_raw/lexicon/staging"
wn.config.data_directory = str(data_fol / "_raw/lexicon/wn_data")


def staged(dataset: str, lang: str) -> pd.DataFrame:
    """Read a staged parquet (``<staging>/<dataset>/<lang>.parquet``)."""
    return pq.read_table(staging / dataset / f"{lang}.parquet").to_pandas()


def wordnet(lang: str) -> wn.Wordnet:
    """Open the pinned OMW lexicon for a language (one wordnet, no merging)."""
    return wn.Wordnet(lexicon=f"{OMW_LEXICONS[lang]}:{OMW_VERSION}")


def ili_of(synset: wn.Synset) -> str | None:
    """Return the synset's ILI id as a plain string, or ``None``."""
    il = synset.ili
    return getattr(il, "id", il) or None


lg.info("staging at {}", staging)

## Candidate signals per concept

In [ ]:
# Assemble candidate concept-level signals (en): commonness (SemCor), hypernym
# depth (specificity proxy), and lexfile; plus the per-lemma frequency overlay.
en = wordnet("en")
ili_count: dict[str, int] = defaultdict(int)
ili_depth: dict[str, int] = {}
ili_lemmas: dict[str, set[str]] = defaultdict(set)
for s in en.synsets():
    il = ili_of(s)
    if not il:
        continue
    for se in s.senses():
        c = se.counts()
        ili_count[il] += sum(c) if c else 0
    paths = s.hypernym_paths()
    ili_depth[il] = min((len(p) for p in paths), default=0)
    for lem in s.lemmas():
        ili_lemmas[il].add(lem.lower())

freq_en = staged("frequency", "en")
zipf_en = dict(zip(freq_en["word"].str.lower(), freq_en["zipf"], strict=False))
print("concepts:", len(ili_count), "with depth:", sum(1 for d in ili_depth.values() if d))

## Validate against Kelly CEFR (en)

In [ ]:
# Validate against Kelly CEFR (en): map concept -> its en lemmas -> easiest CEFR
# band, and the strongest single signal (lemma zipf).
cefr_en = staged("cefr", "en")
order = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
cefr_map = {
    str(w).lower(): order[lev]
    for w, lev in zip(cefr_en["word"], cefr_en["level"], strict=False)
    if lev in order
}

rows = []
for il, lemmas in ili_lemmas.items():
    zs = [zipf_en[lem] for lem in lemmas if lem in zipf_en]
    levs = [cefr_map[lem] for lem in lemmas if lem in cefr_map]
    if zs and levs:
        rows.append(
            {
                "zipf": max(zs),
                "log_common": np.log1p(ili_count[il]),
                "depth": ili_depth.get(il, 0),
                "cefr": min(levs),
            }
        )
df = pd.DataFrame(rows)
print("concepts matched to CEFR:", len(df))
df.corr(numeric_only=True)["cefr"].sort_values()

## Cross-language hypothesis (en-easy -> it frequency)

In [ ]:
# Cross-language hypothesis: concepts that are "easy" in en (high commonness),
# are their it lemmas also high-frequency? Quantify disagreement.
freq_it = staged("frequency", "it")
zipf_it = dict(zip(freq_it["word"].str.lower(), freq_it["zipf"], strict=False))
# top-decile commonness among concepts that have any SemCor count (threshold once).
positive = [c for c in ili_count.values() if c > 0]
threshold = np.percentile(positive, 90)
easy = {il for il, c in ili_count.items() if c >= threshold}
hits = miss = 0
for s in wordnet("it").synsets():
    il = ili_of(s)
    if il not in easy:
        continue
    zs = [zipf_it[lem.lower()] for lem in s.lemmas() if lem.lower() in zipf_it]
    if not zs:
        continue
    if max(zs) >= 4.0:  # zipf >= 4 is a common word
        hits += 1
    else:
        miss += 1
print(f"en-easy concepts with an it lemma: {hits + miss}; "
      f"also common in it: {hits} ({100 * hits / (hits + miss):.0f}%), "
      f"disagree: {miss}")

## Findings (measured 2026-06-21)

- **Frequency is the strongest single difficulty signal.** Against Kelly CEFR
  (en, 24,595 matched concepts), max lemma zipf correlates **-0.66** with the
  CEFR band (higher frequency -> easier). SemCor commonness and hypernym depth
  add weaker, partly redundant signal in the same direction.
- **The concept-level part carries most of it, and it travels.** Concepts that
  are easy in English (top-decile commonness) are also high-frequency in Italian
  in the large majority of cases, confirming the brief: difficulty is mostly a
  concept property with a thin per-language overlay.

**Decision (routes to phase 6):** compute complexity mostly at the concept level
(commonness + depth + lexfile, propagated via ILI) and adjust per language with
lemma frequency rank / word length. Validate against Kelly for en / it only;
pt / es / fr stay estimated (no graded list). Kelly is CC-BY-NC-SA, used for
validation only and never shipped.